# Linear Regression and Gradient Descent

## Objective

This notebook presents several regression exercises using Python and Scikit-learn:

- Multiple Linear Regression for house price prediction
- Multiple Linear Regression for salary prediction
- Gradient Descent implemented from scratch
- Comparison between Gradient Descent and Scikit-learn Linear Regression


## 1. Import Libraries


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression


## 2. House Price Prediction

The goal is to predict house prices using three features:

- `area`
- `bedrooms`
- `age`


### 2.1 Load the Dataset


In [ ]:
house_df = pd.read_csv("homeprices_1.csv")
house_df


### 2.2 Check Missing Values


In [ ]:
house_df.isnull().sum()


### 2.3 Handle Missing Bedroom Values

Missing values in `bedrooms` are replaced with the floor of the mean number of bedrooms.


In [ ]:
bedrooms_median = np.floor(house_df["bedrooms"].mean())
house_df["bedrooms"] = house_df["bedrooms"].fillna(bedrooms_median)

house_df


### 2.4 Train the Multiple Linear Regression Model


In [ ]:
house_model = LinearRegression()

house_model.fit(
    house_df[["area", "bedrooms", "age"]],
    house_df["price"]
)


### 2.5 Model Parameters


In [ ]:
print("Coefficients:", house_model.coef_)
print("Intercept:", house_model.intercept_)


### 2.6 Make a Prediction


In [ ]:
new_house = pd.DataFrame({
    "area": [3600],
    "bedrooms": [3],
    "age": [30]
})

predicted_price = house_model.predict(new_house)[0]

print(f"Predicted house price: ${predicted_price:,.2f}")


## 3. Salary Prediction

This example predicts salary using:

- years of experience
- test score
- interview score


### 3.1 Load the Dataset


In [ ]:
salary_df = pd.read_csv("salary_data.csv")
salary_df


### 3.2 Check Missing Values


In [ ]:
salary_df.isnull().sum()


### 3.3 Clean the Data


In [ ]:
# Known missing experience values from the exercise
salary_df.loc[0, "experience"] = "one"
salary_df.loc[1, "experience"] = "six"

mean_test_score = np.floor(
    salary_df["test_score(out of 10)"].mean()
)

salary_df["test_score(out of 10)"] = (
    salary_df["test_score(out of 10)"].fillna(mean_test_score)
)

experience_mapping = {
    "one": 1,
    "two": 2,
    "three": 3,
    "five": 5,
    "six": 6,
    "seven": 7,
    "ten": 10,
    "eleven": 11
}

salary_df["experience"] = salary_df["experience"].map(experience_mapping)

salary_df


### 3.4 Train the Salary Model


In [ ]:
salary_model = LinearRegression()

salary_model.fit(
    salary_df[
        [
            "experience",
            "test_score(out of 10)",
            "interview_score(out of 10)"
        ]
    ],
    salary_df["salary($)"]
)


### 3.5 Model Parameters


In [ ]:
print("Coefficients:", salary_model.coef_)
print("Intercept:", salary_model.intercept_)


### 3.6 Salary Predictions


In [ ]:
candidate_1 = pd.DataFrame({
    "experience": [1],
    "test_score(out of 10)": [5],
    "interview_score(out of 10)": [3]
})

candidate_2 = pd.DataFrame({
    "experience": [1],
    "test_score(out of 10)": [9],
    "interview_score(out of 10)": [8]
})

print(
    f"Candidate 1 predicted salary: "
    f"${salary_model.predict(candidate_1)[0]:,.2f}"
)

print(
    f"Candidate 2 predicted salary: "
    f"${salary_model.predict(candidate_2)[0]:,.2f}"
)


## 4. Gradient Descent from Scratch

The following function estimates the slope `m` and intercept `b`
of a linear model using Gradient Descent.


In [ ]:
def gradient_descent(x, y, learning_rate=0.01, iterations=1000):
    m = 0.0
    b = 0.0
    n = len(x)

    for _ in range(iterations):
        y_pred = m * x + b

        dm = -(2 / n) * np.sum(x * (y - y_pred))
        db = -(2 / n) * np.sum(y - y_pred)

        m -= learning_rate * dm
        b -= learning_rate * db

    y_pred = m * x + b
    cost = np.mean((y - y_pred) ** 2)

    return m, b, cost


### 4.1 Test Gradient Descent


In [ ]:
x_demo = np.array([1, 2, 3, 4, 5], dtype=float)
y_demo = np.array([5, 7, 9, 11, 13], dtype=float)

m, b, cost = gradient_descent(x_demo, y_demo)

print(f"Slope: {m:.4f}")
print(f"Intercept: {b:.4f}")
print(f"Final cost: {cost:.6f}")


In [ ]:
plt.scatter(x_demo, y_demo, label="Actual data")
plt.plot(x_demo, m * x_demo + b, label="Gradient Descent line")

plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear Regression Using Gradient Descent")
plt.legend()
plt.show()


## 5. Student Scores: Gradient Descent vs Scikit-learn

This section compares a manually implemented Gradient Descent model
with Scikit-learn's `LinearRegression`.


### 5.1 Load the Dataset


In [ ]:
scores_df = pd.read_csv("test_scores.csv")
scores_df


### 5.2 Scikit-learn Linear Regression


In [ ]:
score_model = LinearRegression()

score_model.fit(
    scores_df[["math"]],
    scores_df["cs"]
)

print("Coefficient:", score_model.coef_[0])
print("Intercept:", score_model.intercept_)


### 5.3 Gradient Descent


In [ ]:
def gradient_descent_scores(
    x,
    y,
    learning_rate=0.0002,
    iterations=1_000_000,
    tolerance=1e-12
):
    m = 0.0
    b = 0.0
    n = len(x)
    previous_cost = float("inf")

    for i in range(iterations):
        y_pred = m * x + b
        cost = np.mean((y - y_pred) ** 2)

        dm = -(2 / n) * np.sum(x * (y - y_pred))
        db = -(2 / n) * np.sum(y - y_pred)

        m -= learning_rate * dm
        b -= learning_rate * db

        if math.isclose(cost, previous_cost, rel_tol=tolerance):
            break

        previous_cost = cost

    return m, b, cost, i + 1


x_scores = scores_df["math"].to_numpy(dtype=float)
y_scores = scores_df["cs"].to_numpy(dtype=float)

gd_m, gd_b, gd_cost, gd_iterations = gradient_descent_scores(
    x_scores,
    y_scores
)

print(f"Gradient Descent slope: {gd_m:.6f}")
print(f"Gradient Descent intercept: {gd_b:.6f}")
print(f"Final cost: {gd_cost:.6f}")
print(f"Iterations: {gd_iterations}")


### 5.4 Compare Predictions


In [ ]:
comparison = scores_df.copy()

comparison["sklearn_prediction"] = score_model.predict(
    scores_df[["math"]]
)

comparison["gradient_descent_prediction"] = (
    gd_m * x_scores + gd_b
)

comparison


### 5.5 Visualize the Regression Line


In [ ]:
plt.scatter(
    scores_df["math"],
    scores_df["cs"],
    label="Actual data"
)

plt.plot(
    scores_df["math"],
    score_model.predict(scores_df[["math"]]),
    label="Scikit-learn regression"
)

plt.xlabel("Math Score")
plt.ylabel("Computer Science Score")
plt.title("Linear Regression - Student Scores")
plt.legend()
plt.show()


## Conclusion

This notebook demonstrated how Linear Regression can be used for different prediction tasks.

It also showed how Gradient Descent can be implemented manually and compared with
Scikit-learn's `LinearRegression` model.
